In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter
from coverage_functions import coverage_calculator, plot_time_series, plot_time_spacing


Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r before_after_details_true
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

In [ ]:
for loc_id in location_ids:
    df = df.assign(item_modifications = lambda df: df['item_modifications'].str.title())

In [ ]:
loc_id = 'LQ5EH4BKGV61T'
df = sales_and_menu_data[loc_id]
promo_datetime = before_after_details_true.loc[loc_id, 'cross_over_date']
before_abf = (df
       .loc[promo_datetime - pd.DateOffset(months=2):promo_datetime]
       .query('is_plant_based == "No"')
       ['item_quantity']
       .sum())
after_abf = (df
       .loc[promo_datetime:promo_datetime + pd.DateOffset(months=2)]
       .query('is_plant_based == "No"')
       ['item_quantity']
       .sum())
before_abf, after_abf

In [ ]:
before_after_details_true

In [ ]:
import instrumental_programs as iv
import scipy.optimize as opt
import numpy as np


def test_iv():
    graph = [('Z', 'A'), ('A', 'Y')]
    instruments = {'Z'}
    measurements = {'A', 'Y'}
    unobserved = set()
    cardinalities = {'Z': 2, 'A': 2, 'Y': 2}

    dist = np.array([[[0.125, 0.125], [0.125, 0.125]], [[0.125, 0.125], [0.125, 0.125]]])
    dist_dims = ['Z', 'A', 'Y']

    assumptions = [
        {
            'type': 'positive_effect',
            'parent': 'A',
            'child': 'Y',
            'kwargs': {
                'epsilon': 0.01
            },
        },
    ]

    linprog_args = iv.build_lp(graph,
                               cardinalities,
                               instruments,
                               measurements,
                               unobserved,
                               dist,
                               dist_dims,
                               target_var='Y',
                               intervention={'A': 0},
                               intervention2={'A': 1},
                               assumptions=assumptions,
                               n=1000,
                               alpha=0.05)
    #print(linprog_args)
    print(opt.linprog(*linprog_args))

test_iv()